In [2]:
#!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [4]:
from langchain_core.documents import Document


In [15]:
sample_doc=Document(
    page_content="This is a sample",
    metadata={"source":"https//google.com"}
)

In [16]:
sample_doc

Document(metadata={'source': 'https//google.com'}, page_content='This is a sample')

In [17]:
type(sample_doc)

langchain_core.documents.base.Document

In [29]:
#Text loader
from langchain_community.document_loaders.text import TextLoader

loader=TextLoader("data/Python.txt",encoding="utf-8")

In [30]:
document=loader.load()

In [31]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [34]:
from langchain_community.document_loaders.pdf import PyPDFLoader

pdf_loader=PyPDFLoader("data/research2.pdf")

In [35]:
document=pdf_loader.load()

In [38]:
#document

# Ingeston Pipeline

### Documents

In [40]:
import os

In [51]:
def load_all_pdfs():
    folder_path = "data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdfpath = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdfpath)
            doc = loader.load()

            num_docs += 1
            all_docs.extend(doc)

    print("all pdf", num_docs)
    print("total pages", len(all_docs))

    return all_docs

In [52]:
all_docs=load_all_pdfs()

all pdf 2
total pages 32


### Chunks

In [54]:
#!pip install langchain_text_splitters

In [64]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=500,chunk_overlap=50):

    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs=text_splitter.split_documents(documents)
    return chunked_docs

In [65]:
chunks=split_docs(all_docs)

In [66]:
len(chunks)

#each chunk contain 500 character
#each page has 10 chunks


321

### Embeddings

In [68]:
from sentence_transformers import SentenceTransformer

In [69]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("loading  model",self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("embedding dimenston=",self.model.get_sentence_embedding_dimension())

    def generate_embedding(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("embedding shape=",embeddings.shape)
        return embeddings

In [71]:
embedding_manager=EmbeddingManager()

loading  model all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimenston= 384


C:\Users\amank\AppData\Local\Temp\ipykernel_12596\654680101.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimenston=",self.model.get_sentence_embedding_dimension())


### Vector store

In [72]:
import chromadb
import uuid

In [84]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [85]:
vector_store=VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [86]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape= (321, 384)
total documents added in vector store= 321
docs in collection: 321


# Retrival Pipeline

In [87]:
from sklearn.metrics.pairwise import cosine_similarity

In [91]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embedding([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [92]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [93]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape= (1, 384)
retrieved 4 documents


[{'id': 'doc_ed1df36a-7416-49bd-8888-86322fb34451',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7\namankumar105541@gmail.com',
  'metadata': {'firstpage': '5998',
   'book': 'Advances in Neural Information Processing Systems 30',
   'created': '2017',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'producer': 'pdfcpu v0.12.1 dev',
   'title': 'Attention is All you Need',
   'page_label': '7',
   'page': 6,
   'content_length': 138,
   'creator': 'PyPDF',
   'total_pages': 11,
   'eventtype': 'Poster',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'date': '2017',
   'creationdate': '2026-09-05T08:10:09+00:00',
   'published': '2017',
   'moddate': '2026-09-05T08:10:09+00:00',
   'publisher': 'Curran Associates, Inc.',
   'source': 'data\\research1.pdf',
   'description-abstract': 'The 